# Imports

In [1]:
import pyspark.sql.functions as f
from pyspark.sql import SparkSession
from pyspark import SparkConf

In [2]:
import numpy as np
import pandas as pd


from string import ascii_lowercase

In [ ]:
parameters = {
    "spark.driver.maxResultSize": "3g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.driver.memory": "40g",
}

spark_conf = SparkConf().setAll(parameters.items())
spark = SparkSession.builder.appName('New').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/05 22:19:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 51810)
Traceback (most recent call last):
  File "/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/socketserver.py", line 766, in __init__
    self.handle()
  File "/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/pyth

# Read Data

In [4]:
int_customers = spark.read.parquet("../data/02_intermediate/int_customers")
int_vehicles = spark.read.parquet("../data/02_intermediate/int_vehicles")
int_transactions = spark.read.parquet("../data/02_intermediate/int_transactions")

In [5]:
customers_info = int_customers
vehicles_info = int_vehicles
transactions_info = int_transactions
start_dt = pd.to_datetime('2020-07-01')
end_dt = pd.to_datetime('2026-07-01')
frequency = '1M'
customers_to_remove = None

In [6]:
def create_auxillary_columns(
    df,
    unit_of_analysis,
    time_column,
):
    """Return a DataFrame with patterned '_id' and '_observ_end_dt' columns.

    Args:
        df (pyspark.sql.DataFrame): The input DataFrame.
        unit_of_analysis (str, optional): The column used for '_id' pattern. Default is None.
        time_column (str, optional): The column used for '_observ_end_dt' pattern. Default is None.

    Returns:
        pyspark.sql.DataFrame: A DataFrame with '_id' and '_observ_end_dt' columns as specified.

    Examples:
        Usage examples of the create_auxiliary_columns function:

        1. Add '_id' and '_observ_end_dt' columns based on a unit of analysis and a time column:
        ```python
        from pyspark.sql import SparkSession

        spark = SparkSession.builder.getOrCreate()

        data = [(1, '2023-01-01'), (2, '2023-02-01'), (3, '2023-03-01')]
        columns = ['unit_id', 'date']

        df = spark.createDataFrame(data, columns)

        result_df = create_auxiliary_columns(df, unit_of_analysis='unit_id', time_column='date')
        result_df.show()
        ```

        2. Add only '_id' column based on a unit of analysis:
        ```python
        from pyspark.sql import SparkSession

        spark = SparkSession.builder.getOrCreate()

        data = [(1, '2023-01-01'), (2, '2023-02-01'), (3, '2023-03-01')]
        columns = ['unit_id', 'date']

        df = spark.createDataFrame(data, columns)

        result_df = create_auxiliary_columns(df, unit_of_analysis='unit_id')
        result_df.show()
        ```

        3. Add only '_observ_end_dt' column based on a time column:
        ```python
        from pyspark.sql import SparkSession

        spark = SparkSession.builder.getOrCreate()

        data = [(1, '2023-01-01'), (2, '2023-02-01'), (3, '2023-03-01')]
        columns = ['unit_id', 'date']

        df = spark.createDataFrame(data, columns)

        result_df = create_auxiliary_columns(df, time_column='date')
        result_df.show()
        ```

        4. No additional columns added (both unit_of_analysis and time_column are None):
        ```python
        from pyspark.sql import SparkSession

        spark = SparkSession.builder.getOrCreate()

        data = [(1, '2023-01-01'), (2, '2023-02-01'), (3, '2023-03-01')]
        columns = ['unit_id', 'date']

        df = spark.createDataFrame(data, columns)

        result_df = create_auxiliary_columns(df)
        result_df.show()
        ```
    """
    if unit_of_analysis:
        if len(unit_of_analysis) < 2:
            df = df.withColumn(
                "_id", 
                f.col(unit_of_analysis)
            )
        else:
            df = df.withColumn(
                "_id", 
                f.concat_ws("__", *unit_of_analysis)
            )

    if time_column:
        df = df.withColumn(
            "_observ_end_dt",
            f.last_day(f.col(time_column))
            # f.to_date(f.date_trunc("quarter", f.col(time_column)))
        )

    return df


# Create Spine

In [7]:
# Create range dataframe
range_of_dates = pd.DataFrame()
range_of_dates["_observ_end_dt"] = pd.date_range(start=start_dt, end=end_dt, freq=frequency)

# Create a DataFrame with the range of dates
range_of_dates = spark.createDataFrame(range_of_dates).withColumn(
    "_observ_end_dt", f.date_format("_observ_end_dt", "yyyy-MM-dd")
).withColumn(
    "_observ_end_dt",
    f.to_date(f.last_day(f.col("_observ_end_dt")))
    # f.to_date(f.date_trunc("quarter", f.col("_observ_end_dt")))
)

/var/folders/3r/sqxygt_d4gs0bc8yhzxh19vw0000gp/T/ipykernel_14389/4188962588.py:3: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  range_of_dates["_observ_end_dt"] = pd.date_range(start=start_dt, end=end_dt, freq=frequency)
/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  sun.misc.Unsafe or java.nio.DirectByteBuffer.<init>(long, int) not available
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


In [8]:
# Get the distinct customers from customers_info
customers_info = customers_info.select("customer_id", "mobile").distinct()
vehicles_info = vehicles_info.select("customer_id", "customer_vehicle_id", "plate_number").distinct()

customer_universe_agg  = vehicles_info.groupBy(
    "customer_id"
).agg(
    f.countDistinct("customer_vehicle_id").alias("count")
)

customer_universe = customers_info.join(
    vehicles_info,
    on="customer_id",
    how="inner",
).join(
    customer_universe_agg.filter(
        f.col("count") < 11
    ),
    on="customer_id",
    how="inner",
).drop("count")

# Filter out customers to remove
if customers_to_remove is not None:
    customer_universe = customer_universe.filter(~f.col("customer_id").isin(customers_to_remove))

In [9]:
wrong_plates = [
    f"{number}"*4 + f"{letter.upper()}"*3 
    for letter in ascii_lowercase 
    for number in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
]

wrong_mobiles = [
    f"0{number1}" + f"{number2}"*8 
    for number1 in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] 
    for number2 in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
]

In [10]:
customer_universe_info = customer_universe.filter(
    f.length("plate_number") == 7
).filter(
    f.length("mobile") == 10
).filter(
    ~f.col("plate_number").isin(wrong_plates)
).filter(
    ~f.col("mobile").isin(wrong_mobiles)
).withColumn(
    "mobile",
    f.overlay(f.col("mobile"), f.lit("966"), 0, 2)
).withColumn(
    "_id", 
    f.concat_ws("__", "plate_number", "mobile")
).withColumn(
    "_key", 
    f.concat_ws("__", "customer_id", "customer_vehicle_id")
)

customer_universe_selected = customer_universe_info.select("_id", "_key")

In [11]:
transactions_info = create_auxillary_columns(
    transactions_info,
    unit_of_analysis=["customer_id", "customer_vehicle_id"],
    time_column="transaction_dt"
).withColumnRenamed("_id", "_key")

transactions_info_agg = transactions_info.groupBy(
    "_key"
).agg(
    f.min("_observ_end_dt").alias("first_transaction_period")
)

transactions_info_agg = transactions_info_agg.join(
    customer_universe_selected,
    on="_key",
    how="inner"
).groupBy(
    "_id"
).agg(
    f.min("first_transaction_period").alias("first_transaction_period")
).select("_id", "first_transaction_period")

In [12]:
# Create the spine with a cross join
spine = customer_universe_selected.crossJoin(
    range_of_dates
).join(
    transactions_info_agg,
    on="_id",
    how="left"
).filter(
    f.col("_observ_end_dt") >= f.col("first_transaction_period")
)

# breakpoint()

# Order dataframe by ids and cohort date
spine = spine.select("_id", "_key", "_observ_end_dt").orderBy(["_id", "_key", "_observ_end_dt"])

In [14]:
spine.count()

160355835

# Check audit files

In [5]:
raw_customers = spark.read.parquet("../data/01_raw/raw_customers.parquet")
raw_vehicles = spark.read.parquet("../data/01_raw/raw_vehicles.parquet")

int_customers = spark.read.parquet("../data/02_intermediate/int_customers")
int_vehicles = spark.read.parquet("../data/02_intermediate/int_vehicles")
int_transactions = spark.read.parquet("../data/02_intermediate/int_transactions")

prm_customers = spark.read.parquet("../data/03_primary/prm_customers")
prm_vehicles = spark.read.parquet("../data/03_primary/prm_vehicles")

In [6]:
june_audit_df = pd.read_csv("mileage_model_audit/june_audit.csv", sep=";")

/var/folders/3r/sqxygt_d4gs0bc8yhzxh19vw0000gp/T/ipykernel_21049/3807196400.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  june_audit_df = pd.read_csv("mileage_model_audit/june_audit.csv", sep=";")


In [7]:
june_audit_df["id"] = june_audit_df["Car Number"].astype(str).str.strip().str.zfill(7) + "__" + june_audit_df["Mobile"].astype(str).str.replace(" ", "").str.strip()
june_audit_df.head(2)

,Mobile,Car Number,Customer Type,Customer_Identifier,Invoice Date,Invoice Number,Net_Sales,Last Message Date,Last Message Campaign,Last Message Wave,...,Next Message Wave,Next Message Cohort,group,last mileage,mileage/day,predicted visit date,customer type (min vs synth),historical visits,behavior,id
0,013 882 9996,3907DSB,Winback Customers,013 882 9996_3907DSB,02/06/25,M-1907-001-19508,"88,7",NaN,NaN,NaN,...,NaN,NaN,Test,NaN,NaN,NaN,NaN,NaN,NaN,3907DSB__0138829996
1,966052254203,8893JGD,Existing Customers,966052254203_8893JGD,24/06/25,M-2258-012-24130,"52,17",NaN,NaN,NaN,...,NaN,NaN,Control,NaN,NaN,NaN,NaN,NaN,NaN,8893JGD__966052254203


In [8]:
june_audit_df["id"].shape, june_audit_df["id"].drop_duplicates().shape

((242363,), (229830,))

In [9]:
june_audit_df["Mobile"].shape, june_audit_df["Mobile"].drop_duplicates().shape

((242363,), (218955,))

In [10]:
june_audit_df["Car Number"].drop_duplicates().str.len().value_counts()

Car Number
7    223915
6      3751
5      1368
4       541
8        13
3         4
Name: count, dtype: int64

In [11]:
mobile_list = sorted(set(june_audit_df["Mobile"].astype(str).str.replace(" ", "").str.strip().to_list()))
plate_list = sorted(set(june_audit_df["Car Number"].astype(str).str.strip().str.zfill(7).to_list()))
id_list = sorted(set(june_audit_df["id"].to_list()))

In [12]:
len(id_list), len(mobile_list), len(plate_list)

(229830, 218955, 229590)

Let's go deeper and check on components of the spine

### Raw

In [13]:
raw_customers.show(2)
raw_vehicles.show(2)

25/07/05 22:20:06 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+--------------------+-------------------+-------------------+---------+------------+----------+-------+-----+------+-----------+---------+------------+-------+--------------+-------+----+--------+------------+
|          CustomerID|         CreationOn|         ModifiedOn|UserTitle|CustomerName|    Mobile|Mobile2|Email|Gender|Nationality|BirthDate|LocationName|IsOwner|IsCashCustomer|PayTerm| Age|IsActive|StationBrand|
+--------------------+-------------------+-------------------+---------+------------+----------+-------+-----+------+-----------+---------+------------+-------+--------------+-------+----+--------+------------+
|EB5670AA-72D8-49C...|1615507878000000000|1585223724000000000|     NULL|   ABU RAKAN|0555551397|   NULL| NULL|  NULL|       NULL|     NULL|        NULL|   true|          true|   NULL|NULL|    true|          PE|
|4C86E939-005E-461...|1615507878000000000|1585223724000000000|     NULL|  MR OLAYYAN|0547502121|   NULL| NULL|  NULL|       NULL|     NULL|        NULL|   t

+--------------------+-------------------+-------------------+--------------------+---------+--------+---------+----------------+-------+-----------+-----------------+-------------+------------+--------+-------------------+
|   CustomerVehicleID|         CreationOn|         ModifiedOn|          CustomerID|     Make|   Model|ModelYear|TransmissionType|PMSName|PlateNumber|              VIN|IsDriverOwner|StationBrand|is_truck|vehicle_brand_level|
+--------------------+-------------------+-------------------+--------------------+---------+--------+---------+----------------+-------+-----------+-----------------+-------------+------------+--------+-------------------+
|918FE112-E062-4A5...|1615507904000000000|1585223724000000000|40AA5E88-1966-449...|chevrolet|suburban|   2013.0|       Automatic|   NULL|    5340AAJ|1GNSC8E07DR219313|        false|          PE|       0|                low|
|1B4AA1DD-1E18-499...|1615507904000000000|1585223724000000000|E1EAFA23-0ADC-4AF...|    lexus|   ls400|  

In [14]:
raw_customers.select("Mobile").distinct().count()

4371748

In [15]:
raw_customers.withColumn(
    "new_mobile",
    f.overlay(f.col("Mobile"), f.lit("966"), 0, 2)
).filter(
    f.col("new_mobile").isin(mobile_list)
).select("new_mobile").distinct().count()

25/07/05 22:24:00 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB
25/07/05 22:24:03 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB


125017

In [16]:
raw_vehicles.select("PlateNumber").distinct().count()

5878414

In [17]:
raw_vehicles.filter(
    f.col("PlateNumber").isin(plate_list)
).select("PlateNumber").distinct().count()

25/07/05 22:25:55 WARN DAGScheduler: Broadcasting large task binary with size 10.3 MiB
25/07/05 22:25:59 WARN DAGScheduler: Broadcasting large task binary with size 10.3 MiB


181600

In [18]:
raw_customers_vehicle = raw_customers.withColumn(
    "new_mobile",
    f.overlay(f.col("Mobile"), f.lit("966"), 0, 2)
).select(
    "CustomerID",
    "Mobile",
    "new_mobile",
).join(
    raw_vehicles.select("CustomerVehicleID", "CustomerID", "PlateNumber"),
    on="CustomerID",
    how="outer",
).withColumn(
    "_id", 
    f.concat_ws("__", "PlateNumber", "new_mobile")
)

raw_customers_vehicle.show(2, truncate=False)

+------------------------------------+----------+------------+------------------------------------+-----------+---------------------+
|CustomerID                          |Mobile    |new_mobile  |CustomerVehicleID                   |PlateNumber|_id                  |
+------------------------------------+----------+------------+------------------------------------+-----------+---------------------+
|00000540-7986-4B85-A946-E1A9D913344F|0534747726|966534747726|F40BF538-A9A6-49E3-9623-9301AE9FD953|1065KHD    |1065KHD__966534747726|
|00000800-7F43-4A99-AD7F-46518BF588D5|0504050603|966504050603|650A8060-A847-45D2-B0EB-07C79BB52A4B|1687RXJ    |1687RXJ__966504050603|
+------------------------------------+----------+------------+------------------------------------+-----------+---------------------+
only showing top 2 rows



In [19]:
raw_customers_vehicle.filter(
    f.contains(f.col("PlateNumber"), f.lit("8893JGD"))
).show(2, truncate=False)

+------------------------------------+----------+------------+------------------------------------+-----------+---------------------+
|CustomerID                          |Mobile    |new_mobile  |CustomerVehicleID                   |PlateNumber|_id                  |
+------------------------------------+----------+------------+------------------------------------+-----------+---------------------+
|115ED86B-9445-4D62-9711-59A7B9D92B3D|0000000000|966000000000|9176DEF9-1B81-4CC8-A311-7D99ECBB7294|8893JGD    |8893JGD__966000000000|
+------------------------------------+----------+------------+------------------------------------+-----------+---------------------+



In [20]:
raw_customers_vehicle.select("new_mobile").distinct().count()

4371749

In [21]:
raw_customers_vehicle.filter(
    f.col("new_mobile").isin(mobile_list)
).select("new_mobile").distinct().count()

25/07/05 22:30:02 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB
25/07/05 22:30:05 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB


125017

In [22]:
raw_customers_vehicle.select("PlateNumber").distinct().count()

5878415

In [23]:
raw_customers_vehicle.filter(
    f.col("PlateNumber").isin(plate_list)
).select("PlateNumber").distinct().count()

25/07/05 22:32:05 WARN DAGScheduler: Broadcasting large task binary with size 10.3 MiB
25/07/05 22:32:08 WARN DAGScheduler: Broadcasting large task binary with size 10.3 MiB


181600

In [24]:
raw_customers_vehicle.select("_id").distinct().count()

6575847

In [25]:
raw_customers_vehicle.filter(
    f.col("_id").isin(id_list)
).select("_id").distinct().count()

25/07/05 22:34:20 WARN DAGScheduler: Broadcasting large task binary with size 7.5 MiB
25/07/05 22:34:25 WARN DAGScheduler: Broadcasting large task binary with size 7.5 MiB


60322

### Int

Nothing should have changed from raw to int - only column renaming

In [26]:
int_customers.show(2)
int_vehicles.show(2)

+--------------------+-----------+-----------+----------+-------------+----------+-------+-----+------+-----------+----------+-------------+--------+----------------+-------+----+---------+-------------+
|         customer_id|creation_on|modified_on|user_title|customer_name|    mobile|mobile2|email|gender|nationality|birth_date|location_name|is_owner|is_cash_customer|payterm| age|is_active|station_brand|
+--------------------+-----------+-----------+----------+-------------+----------+-------+-----+------+-----------+----------+-------------+--------+----------------+-------+----+---------+-------------+
|EB5670AA-72D8-49C...| 2021-03-12| 2020-03-26|      NULL|    ABU RAKAN|0555551397|   NULL| NULL|  NULL|       NULL|1900-01-01|         NULL|    true|            true|   NULL|NULL|     true|           PE|
|4C86E939-005E-461...| 2021-03-12| 2020-03-26|      NULL|   MR OLAYYAN|0547502121|   NULL| NULL|  NULL|       NULL|1900-01-01|         NULL|    true|            true|   NULL|NULL|     

In [27]:
int_customers.select("mobile").distinct().count()

4371748

In [28]:
int_customers.withColumn(
    "new_mobile",
    f.overlay(f.col("mobile"), f.lit("966"), 0, 2)
).filter(
    f.col("new_mobile").isin(mobile_list)
).select("new_mobile").distinct().count()

25/07/05 22:36:22 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB
25/07/05 22:36:24 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB


125017

In [29]:
int_vehicles.select("plate_number").distinct().count()

5878414

In [30]:
int_vehicles.withColumn(
    "new_plate",
    f.lpad("plate_number", 7, "0")
).filter(
    f.col("new_plate").isin(plate_list)
).select(f.countDistinct("new_plate")).show()

int_vehicles.filter(
    f.col("plate_number").isin(plate_list)
).select(f.countDistinct("plate_number")).show()


25/07/05 22:38:30 WARN DAGScheduler: Broadcasting large task binary with size 6.4 MiB
25/07/05 22:38:31 WARN DAGScheduler: Broadcasting large task binary with size 6.4 MiB


+-------------------------+
|count(DISTINCT new_plate)|
+-------------------------+
|                   183764|
+-------------------------+



25/07/05 22:40:30 WARN DAGScheduler: Broadcasting large task binary with size 10.3 MiB
25/07/05 22:40:34 WARN DAGScheduler: Broadcasting large task binary with size 10.3 MiB


+----------------------------+
|count(DISTINCT plate_number)|
+----------------------------+
|                      181600|
+----------------------------+



In [31]:
int_customers_vehicle = int_customers.withColumn(
    "new_mobile",
    f.overlay(f.col("mobile"), f.lit("966"), 0, 2)
).withColumnRenamed(
    "creation_on", "customer_creation_on"
).withColumnRenamed(
    "modified_on", "customer_modified_on"
).select(
    "customer_id",
    "mobile",
    "new_mobile",
    "customer_creation_on",
    "customer_modified_on"
).join(
    int_vehicles.withColumn(
        "new_plate_number",
        f.lpad(f.col("plate_number"), 7, "0")
    ).withColumnRenamed(
        "creation_on", "vehicle_creation_on"
    ).withColumnRenamed(
        "modified_on", "vehicle_modified_on"
    ).select(
        "customer_vehicle_id",
        "customer_id",
        "plate_number",
        "vehicle_creation_on",
        "vehicle_modified_on"
    ),
    on="customer_id",
    how="outer",
).withColumn(
    "_id", 
    f.concat_ws("__", "plate_number", "new_mobile")
)

int_customers_vehicle.show(2, truncate=False)

+------------------------------------+----------+------------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+---------------------+
|customer_id                         |mobile    |new_mobile  |customer_creation_on|customer_modified_on|customer_vehicle_id                 |plate_number|vehicle_creation_on|vehicle_modified_on|_id                  |
+------------------------------------+----------+------------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+---------------------+
|00000540-7986-4B85-A946-E1A9D913344F|0534747726|966534747726|2023-04-11          |2023-04-12          |F40BF538-A9A6-49E3-9623-9301AE9FD953|1065KHD     |2023-04-11         |2023-04-12         |1065KHD__966534747726|
|00000800-7F43-4A99-AD7F-46518BF588D5|0504050603|966504050603|2021-03-12          |2020-03-26          |650A8060-A847-45D2-B0EB-07C7

In [32]:
int_customers_vehicle.count()

7113059

In [33]:
int_customers_vehicle.select("new_mobile").distinct().count()

4371749

In [34]:
int_customers_vehicle.filter(
    f.col("new_mobile").isin(mobile_list)
).select("new_mobile").distinct().count()

25/07/05 22:42:40 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB
25/07/05 22:42:41 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB


125017

In [35]:
int_customers_vehicle.select("plate_number").distinct().count()

5878415

In [36]:
int_customers_vehicle.filter(
    f.col("plate_number").isin(plate_list)
).select("plate_number").distinct().count()

25/07/05 22:44:45 WARN DAGScheduler: Broadcasting large task binary with size 10.3 MiB
25/07/05 22:44:50 WARN DAGScheduler: Broadcasting large task binary with size 10.3 MiB


181600

In [37]:
int_customers_vehicle.select("_id").distinct().count()

6575847

In [38]:
int_customers_vehicle.filter(
    f.col("_id").isin(id_list)
).select("_id").distinct().count()

25/07/05 22:47:00 WARN DAGScheduler: Broadcasting large task binary with size 7.5 MiB
25/07/05 22:47:04 WARN DAGScheduler: Broadcasting large task binary with size 7.5 MiB


60322

### CREATE TABLE

In [39]:
spark = SparkSession.builder.getOrCreate()

In [40]:
june_audit_df["mobile"] = june_audit_df["Mobile"].astype(str).str.strip().str.replace(" ","")
june_audit_df["plate_number"] = june_audit_df["Car Number"]
june_audit_df["_id"] = june_audit_df["id"]

In [41]:
june_audit_df["Mobile"].astype(str).str.strip()

0         013 882 9996
1         966052254203
2         966123456785
3         966123458154
4         966222222222
              ...     
242358    966599999999
242359    966599999999
242360    966599999999
242361    966920002806
242362    971525806092
Name: Mobile, Length: 242363, dtype: object

In [42]:
june_audit_df["mobile"].shape, june_audit_df["mobile"].drop_duplicates().shape

((242363,), (218955,))

In [43]:
june_sdf = spark.createDataFrame(june_audit_df[["_id", "plate_number", "mobile"]])
june_sdf.show(2, truncate=False)

/Users/Andre_Barbosa/anaconda3/envs/petromin/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:351: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  sun.misc.Unsafe or java.nio.DirectByteBuffer.<init>(long, int) not available
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
25/07/05 22:47:07 WARN TaskSetManager: Stage 237 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.


+---------------------+------------+------------+
|_id                  |plate_number|mobile      |
+---------------------+------------+------------+
|3907DSB__0138829996  |3907DSB     |0138829996  |
|8893JGD__966052254203|8893JGD     |966052254203|
+---------------------+------------+------------+
only showing top 2 rows



In [44]:
wrong_plates = [
    f"{number}"*4 + f"{letter.upper()}"*3 
    for letter in ascii_lowercase 
    for number in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
]

wrong_mobiles = [
    f"966{number1}" + f"{number2}"*8 
    for number1 in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] 
    for number2 in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
]

june_sdf = june_sdf.withColumn(
    "valid_mobile",
    f.when(
    #     f.col("mobile").isNull(),
    #     0
    # ).when(
        f.length("mobile") != 12,
        0
    ).when(
        f.col("mobile").isin(wrong_mobiles),
        0
    ).otherwise(1)
).withColumn(
    "valid_plate",
    f.when(
    #     f.col("plate_number").isNull(),
    #     0
    # ).when(
        f.length("plate_number") != 7,
        0
    ).when(
        f.col("plate_number").isin(wrong_plates),
        0
    ).otherwise(1)
).withColumn(
    "valid_id",
    f.when(
        f.col("valid_plate") < 1,
        0
    ).when(
        f.col("valid_mobile") < 1,
        0
    ).otherwise(1)
)

In [45]:
june_sdf.show(20, truncate=False)

june_sdf.select(
    f.countDistinct("_id").alias("unique_id"),
    f.countDistinct("plate_number").alias("unique_plate_number"),
    f.countDistinct("mobile").alias("unique_mobile"),
    f.sum("valid_plate").alias("valid_plate"),
    f.sum("valid_mobile").alias("valid_mobile"),
    f.sum("valid_id").alias("valid_id"),
).show(truncate=False)

june_sdf.drop_duplicates(subset=["_id"]).select(
    f.countDistinct("_id").alias("unique_id"),
    f.countDistinct("plate_number").alias("unique_plate_number"),
    f.countDistinct("mobile").alias("unique_mobile"),
    f.sum("valid_plate").alias("valid_plate"),
    f.sum("valid_mobile").alias("valid_mobile"),
    f.sum("valid_id").alias("valid_id"),
).show(truncate=False)

25/07/05 22:47:10 WARN TaskSetManager: Stage 238 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.
25/07/05 22:47:11 WARN TaskSetManager: Stage 239 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.


+---------------------+------------+------------+------------+-----------+--------+
|_id                  |plate_number|mobile      |valid_mobile|valid_plate|valid_id|
+---------------------+------------+------------+------------+-----------+--------+
|3907DSB__0138829996  |3907DSB     |0138829996  |0           |1          |0       |
|8893JGD__966052254203|8893JGD     |966052254203|1           |1          |1       |
|4489GGD__966123456785|4489GGD     |966123456785|1           |1          |1       |
|6933ZED__966123458154|6933ZED     |966123458154|1           |1          |1       |
|9621NHB__966222222222|9621NHB     |966222222222|0           |1          |0       |
|1486SSD__966232139399|1486SSD     |966232139399|1           |1          |1       |
|0000ARE__966500000000|0000ARE     |966500000000|0           |1          |0       |
|0000RDL__966500000000|0000RDL     |966500000000|0           |1          |0       |
|0000RDL__966500000000|0000RDL     |966500000000|0           |1          |0 

25/07/05 22:47:13 WARN TaskSetManager: Stage 245 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.


+---------+-------------------+-------------+-----------+------------+--------+
|unique_id|unique_plate_number|unique_mobile|valid_plate|valid_mobile|valid_id|
+---------+-------------------+-------------+-----------+------------+--------+
|229830   |229592             |218955       |236345     |242052      |236051  |
+---------+-------------------+-------------+-----------+------------+--------+



+---------+-------------------+-------------+-----------+------------+--------+
|unique_id|unique_plate_number|unique_mobile|valid_plate|valid_mobile|valid_id|
+---------+-------------------+-------------+-----------+------------+--------+
|229830   |229591             |218955       |224130     |229602      |223914  |
+---------+-------------------+-------------+-----------+------------+--------+



In [46]:
from pyspark.sql import Window

w = Window.orderBy("_id")

In [47]:
june_sdf = june_sdf.withColumn(
    "index",
    f.row_number().over(w)
)

june_sdf.show(10)

25/07/05 22:47:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:16 WARN TaskSetManager: Stage 255 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+------------+------------+------------+-----------+--------+-----+
|                 _id|plate_number|      mobile|valid_mobile|valid_plate|valid_id|index|
+--------------------+------------+------------+------------+-----------+--------+-----+
|000036F__96656273...|         36F|966562730605|           1|          0|       0|    1|
|000036F__96656273...|         36F|966562730605|           1|          0|       0|    2|
|00004KH__96650376...|         4KH|966503765736|           1|          0|       0|    3|
|00005KH__96654212...|         5KH|966542126095|           1|          0|       0|    4|
|0000959__96654411...|         959|966544111448|           1|          0|       0|    5|
|0000ARE__96650000...|     0000ARE|966500000000|           0|          1|       0|    6|
|0000JJJ__96650351...|     0000JJJ|966503517692|           1|          0|       0|    7|
|0000JJJ__96650351...|     0000JJJ|966503517692|           1|          0|       0|    8|
|0000JJJ__96650351...

In [48]:
june_sdf.count()

25/07/05 22:47:16 WARN TaskSetManager: Stage 257 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.


242363

In [49]:
june_id_data = june_sdf.join(
    int_customers_vehicle.withColumnRenamed(
        "plate_number", "plate_number_id_db"
    ).withColumnRenamed(
        "mobile", "mobile_id_db"
    ),
    on=["_id"],
    how="left"
)

june_id_data.orderBy("_id").show(100, truncate=False)

june_plate_data = june_sdf.join(
    int_customers_vehicle.withColumnRenamed(
        "_id", "id_db"
    ).withColumnRenamed(
        "new_mobile", "mobile_plate_db"
    ).select("plate_number", "customer_vehicle_id", "vehicle_creation_on", "vehicle_modified_on", "mobile_plate_db").distinct(),
    on=["plate_number"],
    how="left"
).withColumn(
    "is_correct_mobile_on_plate_db",
    f.when(
        f.col("mobile") == f.col("mobile_plate_db"),
        1
    ).otherwise(0)
)

june_plate_data.orderBy("_id").show(100, truncate=False)

june_mobile_data = june_sdf.filter(
    f.col("valid_mobile") == 1
).join(
    int_customers_vehicle.withColumnRenamed(
        "_id", "id_db"
    ).withColumnRenamed(
        "plate_number", "plate_number_mobile_db"
    ).withColumnRenamed(
        "mobile", "mobile_db"
    ).withColumnRenamed(
        "new_mobile", "mobile"
    ).select("mobile", "customer_id", "customer_creation_on", "customer_modified_on", "plate_number_mobile_db").distinct(),
    on=["mobile"],
    how="left"
).withColumn(
    "is_correct_plate_on_mobile_db",
    f.when(
        f.col("plate_number") == f.col("plate_number_mobile_db"),
        1
    ).otherwise(0)
)

june_mobile_data.orderBy("_id").show(100, truncate=False)

25/07/05 22:47:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:17 WARN TaskSetManager: Stage 260 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.
25/07/05 22:47:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:24 WARN WindowExe

+---------------------+------------+------------+------------+-----------+--------+-----+------------------------------------+------------+------------+--------------------+--------------------+------------------------------------+------------------+-------------------+-------------------+
|_id                  |plate_number|mobile      |valid_mobile|valid_plate|valid_id|index|customer_id                         |mobile_id_db|new_mobile  |customer_creation_on|customer_modified_on|customer_vehicle_id                 |plate_number_id_db|vehicle_creation_on|vehicle_modified_on|
+---------------------+------------+------------+------------+-----------+--------+-----+------------------------------------+------------+------------+--------------------+--------------------+------------------------------------+------------------+-------------------+-------------------+
|000036F__966562730605|36F         |966562730605|1           |0          |0       |1    |NULL                                |N

25/07/05 22:47:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:36 WARN TaskSetManager: Stage 293 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.


+------------+---------------------+------------+------------+-----------+--------+-----+------------------------------------+-------------------+-------------------+---------------+-----------------------------+
|plate_number|_id                  |mobile      |valid_mobile|valid_plate|valid_id|index|customer_vehicle_id                 |vehicle_creation_on|vehicle_modified_on|mobile_plate_db|is_correct_mobile_on_plate_db|
+------------+---------------------+------------+------------+-----------+--------+-----+------------------------------------+-------------------+-------------------+---------------+-----------------------------+
|36F         |000036F__966562730605|966562730605|1           |0          |0       |1    |NULL                                |NULL               |NULL               |NULL           |0                            |
|36F         |000036F__966562730605|966562730605|1           |0          |0       |2    |NULL                                |NULL               |NU

25/07/05 22:47:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/05 22:47:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+------------+---------------------+------------+------------+-----------+--------+-----+------------------------------------+--------------------+--------------------+----------------------+-----------------------------+
|mobile      |_id                  |plate_number|valid_mobile|valid_plate|valid_id|index|customer_id                         |customer_creation_on|customer_modified_on|plate_number_mobile_db|is_correct_plate_on_mobile_db|
+------------+---------------------+------------+------------+-----------+--------+-----+------------------------------------+--------------------+--------------------+----------------------+-----------------------------+
|966562730605|000036F__966562730605|36F         |1           |0          |0       |1    |55D5E721-01B8-463E-830B-8BE9FEA942E2|2022-04-16          |2022-04-17          |5404BUB               |0                            |
|966562730605|000036F__966562730605|36F         |1           |0          |0       |1    |3DF012A5-6150-4969-A104

In [50]:
june_mobile_data.count(), june_plate_data.count(), june_id_data.count()

25/07/05 22:47:47 WARN TaskSetManager: Stage 312 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.
25/07/05 22:47:59 WARN TaskSetManager: Stage 335 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.
25/07/05 22:48:09 WARN TaskSetManager: Stage 358 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.


(925808, 313989, 250841)

In [52]:
june_id_data.select(
    f.countDistinct("plate_number_id_db")
).show()

25/07/05 22:52:18 WARN TaskSetManager: Stage 375 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.


+----------------------------------+
|count(DISTINCT plate_number_id_db)|
+----------------------------------+
|                             60306|
+----------------------------------+



In [53]:
june_mobile_data.select(
    f.sum("is_correct_plate_on_mobile_db")
).show()

june_plate_data.select(
    f.sum("is_correct_mobile_on_plate_db")
).show()

25/07/05 22:52:27 WARN TaskSetManager: Stage 399 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.
25/07/05 22:52:37 WARN TaskSetManager: Stage 422 contains a task of very large size (1154 KiB). The maximum recommended task size is 1000 KiB.


+----------------------------------+
|sum(is_correct_plate_on_mobile_db)|
+----------------------------------+
|                             71558|
+----------------------------------+



+----------------------------------+
|sum(is_correct_mobile_on_plate_db)|
+----------------------------------+
|                             71619|
+----------------------------------+



In [ ]:
# +--------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------+---------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------+
# |sum(CASE WHEN ((is_correct_plate_on_mobile_db > 0) AND (is_correct_mobile_on_plate_db > 0)) THEN 1 ELSE 0 END)|sum(CASE WHEN ((n_customer_id_mobile_db > 0) AND (is_correct_plate_on_mobile_db < 1)) THEN 1 ELSE 0 END)|sum(CASE WHEN ((n_customer_vehicle_id_plate_db > 0) AND (is_correct_mobile_on_plate_db < 1)) THEN 1 ELSE 0 END)|sum(CASE WHEN (n_customer_id_mobile_db > 0) THEN 1 ELSE 0 END)|sum(CASE WHEN (n_customer_vehicle_id_plate_db > 0) THEN 1 ELSE 0 END)|sum(CASE WHEN ((is_correct_plate_on_mobile_db < 1) AND (is_correct_mobile_on_plate_db < 1)) THEN 1 ELSE 0 END)|
# +--------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------+---------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------+
# |63100                                                                                                         |77984                                                                                                   |130655                                                                                                         |141084                                                        |193802                                                               |178952                                                                                                        |


In [97]:
# june_sdf.join(
#     june_id_data,
#     on=["_id", "plate_number", "mobile", "index", "valid_mobile", "valid_plate", "valid_id"],
#     how="left"
# ).groupBy(
#     ["_id", "plate_number", "mobile", "index", "valid_mobile", "valid_plate", "valid_id"]
# ).agg(
#     f.countDistinct("customer_id").alias("n_customer_id"),
#     f.min("customer_creation_on").alias("min_customer_creation_on"),
#     f.min("customer_modified_on").alias("min_customer_modified_on"),
#     f.countDistinct("customer_vehicle_id").alias("n_customer_vehicle_id"),
#     f.min("vehicle_creation_on").alias("min_vehicle_creation_on"),
#     f.min("vehicle_modified_on").alias("min_vehicle_modified_on"),
# ).select(
#     f.sum(
#         f.when(
#             f.col("n_customer_id") > 0,
#             1
#         ).otherwise(0)
#     )
# ).show(5, truncate=False) #.coalesce(1).write.option("header", True).csv("june_data/id_audit.csv", sep=",", mode="overwrite")



june_sdf.join(
    june_plate_data,
    on=["_id", "plate_number", "mobile", "index", "valid_mobile", "valid_plate", "valid_id"],
    how="left"
).join(
    june_mobile_data,
    on=["_id", "plate_number", "mobile", "index", "valid_mobile", "valid_plate", "valid_id"],
    how="left"
).groupBy(
    ["_id", "plate_number", "mobile", "index", "valid_mobile", "valid_plate", "valid_id"]
).agg(
    f.countDistinct("customer_id").alias("n_customer_id_mobile_db"),
    f.min("customer_creation_on").alias("min_customer_creation_on_mobile_db"),
    f.min("customer_modified_on").alias("min_customer_modified_on_mobile_db"),
    f.countDistinct("plate_number_mobile_db").alias("n_plate_number_mobile_db"),
    f.sum("is_correct_plate_on_mobile_db").alias("is_correct_plate_on_mobile_db"),
    f.countDistinct("customer_vehicle_id").alias("n_customer_vehicle_id_plate_db"),
    f.min("vehicle_creation_on").alias("min_vehicle_creation_on_plate_db"),
    f.min("vehicle_modified_on").alias("min_vehicle_modified_on_plate_db"),
    f.countDistinct("mobile_plate_db").alias("n_mobile_plate_db"),
    f.sum("is_correct_mobile_on_plate_db").alias("is_correct_mobile_on_plate_db"),
).withColumn(
    "found_correct_id",
    f.when(
        (f.col("is_correct_plate_on_mobile_db") > 0) & (f.col("is_correct_mobile_on_plate_db") > 0),
        1
    ).otherwise(0)
).withColumn(
    "not_found_correct_id",
    f.when(
        ((f.col("is_correct_plate_on_mobile_db") < 1) | (f.col("is_correct_mobile_on_plate_db") < 1)),
        1
    ).otherwise(0)
).withColumn(
    "nothing",
    f.when(
        (f.col("found_correct_id") < 1) & (f.col("not_found_correct_id") < 1),
        1
    ).otherwise(0)
).withColumn(
    "not_found_correct_id__notvalid_plate_notvalid_mobile",
    f.when(
        (f.col("not_found_correct_id") > 0) & (f.col("valid_plate") < 1) & (f.col("valid_mobile") < 1),
        1
    ).otherwise(0)
).withColumn(
    "not_found_correct_id__valid_plate_notvalid_mobile",
    f.when(
        (f.col("not_found_correct_id") > 0) & (f.col("valid_plate") > 0) & (f.col("valid_mobile") < 1),
        1
    ).otherwise(0)
).withColumn(
    "not_found_correct_id__notvalid_plate_valid_mobile",
    f.when(
        (f.col("not_found_correct_id") > 0) & (f.col("valid_plate") < 1) & (f.col("valid_mobile") > 0),
        1
    ).otherwise(0)
).withColumn(
    "not_found_correct_id__valid_id",
    f.when(
        (f.col("not_found_correct_id") > 0) & (f.col("valid_plate") > 0) & (f.col("valid_mobile") > 0),
        1
    ).otherwise(0)
).withColumn(
    "not_found_correct_id__exist_mobile_notexist_plate",
    f.when(
        (f.col("not_found_correct_id__valid_id") > 0) & (f.col("n_customer_id_mobile_db") > 0) & (f.col("n_customer_vehicle_id_plate_db") < 1),
        1
    ).otherwise(0)
).withColumn(
    "not_found_correct_id__exist_plate_notexist_mobile",
    f.when(
        (f.col("not_found_correct_id__valid_id") > 0) & (f.col("n_customer_id_mobile_db") < 1) & (f.col("n_customer_vehicle_id_plate_db") > 0),
        1
    ).otherwise(0)
).withColumn(
    "not_found_correct_id__notexist_plate_notexist_mobile",
    f.when(
        (f.col("not_found_correct_id__valid_id") > 0) & (f.col("n_customer_id_mobile_db") < 1) & (f.col("n_customer_vehicle_id_plate_db") < 1),
        1
    ).otherwise(0)
).withColumn(
    "not_found_correct_id__exist_plate_exist_mobile",
    f.when(
        (f.col("not_found_correct_id__valid_id") > 0) & (f.col("n_customer_id_mobile_db") > 0) & (f.col("n_customer_vehicle_id_plate_db") > 0),
        1
    ).otherwise(0)
).filter(
    f.col("found_correct_id") > 0
).show(100, truncate=False) #.coalesce(1).write.option("header", True).csv("june_data/plate_mobile_audit.csv", sep=",", mode="overwrite")

# .select(
#     f.sum("found_correct_id").alias("total_found_correct_id"),
#     f.sum("not_found_correct_id").alias("total_not_found_correct_id"),
#     f.sum("nothing").alias("total_nothing"),
#     f.sum("not_found_correct_id__notvalid_plate_notvalid_mobile").alias("total_not_found_correct_id__notvalid_plate_notvalid_mobile"),
#     f.sum("not_found_correct_id__valid_plate_notvalid_mobile").alias("total_not_found_correct_id__valid_plate_notvalid_mobile"),
#     f.sum("not_found_correct_id__notvalid_plate_valid_mobile").alias("total_not_found_correct_id__notvalid_plate_valid_mobile"),
#     f.sum("not_found_correct_id__valid_id").alias("total_not_found_correct_id__valid_id"),
#     f.sum("not_found_correct_id__notexist_plate_notexist_mobile").alias("total_not_found_correct_id__notexist_plate_notexist_mobile"),
#     f.sum("not_found_correct_id__exist_mobile_notexist_plate").alias("total_not_found_correct_id__exist_mobile_notexist_plate"),
#     f.sum("not_found_correct_id__exist_plate_notexist_mobile").alias("total_not_found_correct_id__exist_plate_notexist_mobile"),
#     f.sum("not_found_correct_id__exist_plate_exist_mobile").alias("total_not_found_correct_id__exist_plate_exist_mobile"),
# )



ConnectionRefusedError: [Errno 61] Connection refused

In [88]:
int_customers_vehicle.filter(
    f.col("plate_number") == "1510NNR"
).show(100, truncate=False)

int_customers.filter(
    f.col("customer_id") == "412D131D-B1F3-49C9-8F2E-A1DF3BCD1249"
).show(100, truncate=False)

int_customers_vehicle.filter(
    f.col("new_mobile") == "966597518584"
).show(100, truncate=False)

+------------------------------------+------+----------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+-------+
|customer_id                         |mobile|new_mobile|customer_creation_on|customer_modified_on|customer_vehicle_id                 |plate_number|vehicle_creation_on|vehicle_modified_on|_id    |
+------------------------------------+------+----------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+-------+
|412D131D-B1F3-49C9-8F2E-A1DF3BCD1249|NULL  |NULL      |NULL                |NULL                |3BAAD27B-83E9-46A7-9545-492148288188|1510NNR     |2025-05-30         |2025-05-30         |1510NNR|
+------------------------------------+------+----------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+-------+

+-----------+-

+------------------------------------+----------+------------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+---------------------+
|customer_id                         |mobile    |new_mobile  |customer_creation_on|customer_modified_on|customer_vehicle_id                 |plate_number|vehicle_creation_on|vehicle_modified_on|_id                  |
+------------------------------------+----------+------------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+---------------------+
|1D41F2B2-03C8-4EFB-A856-21229594E57F|0597518584|966597518584|2025-04-22          |2025-04-22          |C7F7ED5A-9E40-4CDC-90EB-35A7102E299B|4385NER     |2025-04-22         |2025-04-22         |4385NER__966597518584|
|1DF4461E-3458-416B-B1EC-0F662C1ADB77|0597518584|966597518584|2024-03-13          |2025-05-15          |A28FF943-D204-4430-B3A5-599F

In [ ]:
int_customers_vehicle.filter(
    f.col("plate_number") == "1822GDR"
).show(100, truncate=False)

int_customers.filter(
    f.col("customer_id") == "DCF51E0C-D832-48F7-B6EF-999F66423557"
).show(100, truncate=False)


int_customers_vehicle.filter(
    f.col("new_mobile") == "966505282220"
).show(100, truncate=False)

+------------------------------------+------+----------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+-------+
|customer_id                         |mobile|new_mobile|customer_creation_on|customer_modified_on|customer_vehicle_id                 |plate_number|vehicle_creation_on|vehicle_modified_on|_id    |
+------------------------------------+------+----------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+-------+
|DCF51E0C-D832-48F7-B6EF-999F66423557|NULL  |NULL      |NULL                |NULL                |A4BDC070-5A38-4753-BB58-0FB9B3FA6475|1822GDR     |2024-08-04         |2024-08-04         |1822GDR|
+------------------------------------+------+----------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+-------+

+-----------+-

+------------------------------------+----------+------------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+---------------------+
|customer_id                         |mobile    |new_mobile  |customer_creation_on|customer_modified_on|customer_vehicle_id                 |plate_number|vehicle_creation_on|vehicle_modified_on|_id                  |
+------------------------------------+----------+------------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+---------------------+
|515B23BD-909C-44B1-9899-6A7739A2F0A6|0505282220|966505282220|2021-03-12          |2021-05-20          |0B153E17-7BB0-439B-967F-DB6726578879|5415DLB     |2021-03-12         |2020-09-27         |5415DLB__966505282220|
|D3D2C639-B4C4-4C79-8AC9-760840A90BE7|0505282220|966505282220|2021-06-01          |2023-12-09          |1B1250C4-C1D9-4F76-95B0-B916

In [96]:
int_customers_vehicle.filter(
    f.col("plate_number") == "2271BXA"
).show(100, truncate=False)

int_customers.filter(
    f.col("customer_id") == "955B3908-C92E-4C1B-99C3-6D0A4CDF2612"
).show(100, truncate=False)

int_customers_vehicle.filter(
    f.col("new_mobile") == "966534025083"
).show(100, truncate=False)

+------------------------------------+----------+------------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+---------------------+
|customer_id                         |mobile    |new_mobile  |customer_creation_on|customer_modified_on|customer_vehicle_id                 |plate_number|vehicle_creation_on|vehicle_modified_on|_id                  |
+------------------------------------+----------+------------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+---------------------+
|955B3908-C92E-4C1B-99C3-6D0A4CDF2612|0571520304|966571520304|2024-06-27          |2025-04-03          |F47B70FF-7E3A-415C-9645-19E7500760A8|2271BXA     |2024-06-27         |2024-06-27         |2271BXA__966571520304|
+------------------------------------+----------+------------+--------------------+--------------------+----------------------------

+------------------------------------+----------+------------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+---------------------+
|customer_id                         |mobile    |new_mobile  |customer_creation_on|customer_modified_on|customer_vehicle_id                 |plate_number|vehicle_creation_on|vehicle_modified_on|_id                  |
+------------------------------------+----------+------------+--------------------+--------------------+------------------------------------+------------+-------------------+-------------------+---------------------+
|1CBEA678-1D76-4C1B-957B-993272314011|0534025083|966534025083|2024-04-30          |2024-04-30          |A079BEE3-A7D8-466D-9D9A-4BD0371F4E62|6082STB     |2024-04-30         |2024-04-30         |6082STB__966534025083|
|4629713C-5DCF-4FFE-B0CD-9BC8C5CD9339|0534025083|966534025083|2023-11-09          |2023-11-09          |AE172ED4-23F2-45BB-855A-DB29

25/07/09 08:33:58 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 995308 ms exceeds timeout 120000 ms
25/07/09 08:33:58 WARN SparkContext: Killing executors is not supported by current scheduler.
25/07/09 08:51:53 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

### Prm

In [55]:
prm_customers.select("mobile").distinct().count()

4371748

In [56]:
prm_customers.filter(
    f.col("mobile").isin(mobile_list)
).select("mobile").distinct().count()

25/07/02 06:42:32 WARN DAGScheduler: Broadcasting large task binary with size 14.0 MiB
25/07/02 06:42:37 WARN DAGScheduler: Broadcasting large task binary with size 14.0 MiB


125017

In [57]:
prm_vehicles.select("plate_number").distinct().count()

5845195

In [58]:
prm_vehicles.filter(
    f.col("plate_number").isin(plate_list)
).select("plate_number").distinct().count()

25/07/02 06:44:22 WARN DAGScheduler: Broadcasting large task binary with size 10.3 MiB
25/07/02 06:44:29 WARN DAGScheduler: Broadcasting large task binary with size 10.3 MiB


183243

### Spine components

In [ ]:
spine.select("_id").distinct().filter(
    f.col("_id").isin(id_list)
).count()

25/07/02 05:53:41 WARN DAGScheduler: Broadcasting large task binary with size 7.6 MiB
25/07/02 05:53:41 WARN DAGScheduler: Broadcasting large task binary with size 7.6 MiB
25/07/02 05:53:52 WARN DAGScheduler: Broadcasting large task binary with size 7.6 MiB
25/07/02 05:53:56 WARN DAGScheduler: Broadcasting large task binary with size 7.6 MiB
25/07/02 05:53:59 WARN DAGScheduler: Broadcasting large task binary with size 7.6 MiB


49785

# Prepare table

2

25/07/03 13:47:17 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:123)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:688)
	at org.apache.spark.storage.BlockManagerMasterE